# Day 01 — Client Field Data Validation & GIS Reconciliation

## Corporate scenario

The GIS department receives a recurring field-data delivery from **Client XYZ** containing asset inspection and maintenance updates.

Before the GIS production team can use the delivery for editing, mapping, analysis, reporting, or downstream business processes, the incoming records must be validated against the organization's **authoritative GIS asset inventory** and internal data standards.

As the GIS Developer, your task is to build a repeatable **pre-ingestion quality-control gate** that:

- validates the incoming schema,
- standardizes asset IDs,
- enforces domains and numeric ranges,
- detects missing and duplicate IDs,
- identifies unmatched assets,
- rejects stale inspections,
- supports partial updates without erasing valid GIS values,
- distinguishes true changes from no-change records,
- creates a field-level audit trail,
- updates only approved records in a local working feature class,
- and produces a visual Excel QA/QC dashboard.

**Developer principle:** validate first → reconcile → audit → edit.


## Architecture

```text
Client XYZ field delivery
        ↓
Schema + format validation
        ↓
Business-rule validation
        ↓
Normalize IDs + reconcile with authoritative GIS
        ↓
┌──────────────┬────────────┬────────────┬──────────────┐
│ Ready        │ No Change  │ Rejected   │ Unmatched    │
└──────┬───────┴────────────┴────────────┴──────────────┘
       ↓
Change audit
       ↓
Controlled ArcPy update of local working data
       ↓
CSV exception reports + Excel QA/QC dashboard
       ↓
Validated information for GIS production review
```


## Synthetic exercise scale

The reproducible generator creates:

- **1,000 authoritative GIS point assets**
- **300 Client XYZ field records**
- valid changes
- valid no-change records
- partial updates
- stale inspections
- invalid domains
- out-of-range values
- malformed dates
- missing IDs
- duplicate IDs
- unmatched IDs
- ID-formatting problems that should be normalized successfully

The `scenario_expected` column exists only so we can verify the training exercise. A real client delivery would not contain that field.


In [ ]:
from pathlib import Path
import json
import runpy
import sys

import pandas as pd
import arcpy

PROJECT_DIR = Path.cwd().resolve()
EXPECTED_FOLDER = "day-01-excel-gis-update"

if PROJECT_DIR.name != EXPECTED_FOLDER:
    raise RuntimeError(
        f"Launch Jupyter from the '{EXPECTED_FOLDER}' folder. "
        f"Current folder name: {PROJECT_DIR.name}"
    )

print("Project folder check: PASS")


## 1. Define portable project paths

No personal absolute paths are stored in the notebook. Everything is derived from the repository folder.


In [ ]:
DATA_SCRIPT = PROJECT_DIR / "data" / "create_sample_data.py"
SAMPLE_DIR = PROJECT_DIR / "data" / "sample"
CONFIG_PATH = PROJECT_DIR / "config" / "client_xyz_rules.json"
OUTPUT_DIR = PROJECT_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

GEOJSON_PATH = SAMPLE_DIR / "authoritative_assets.geojson"
EXCEL_PATH = SAMPLE_DIR / "xyz_field_delivery.xlsx"

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from update_features import (
    apply_updates_to_feature_class,
    load_rules,
    read_feature_class,
    reconcile_delivery,
    split_reports,
    write_csv_reports,
)
from generate_dashboard import generate_dashboard

print("Project paths configured.")


## 2. Generate reproducible corporate test data

The source files are generated locally and ignored by Git. The generator script is committed, so anyone cloning the repository can recreate the exact same exercise.


In [ ]:
runpy.run_path(str(DATA_SCRIPT), run_name="__main__")

assert GEOJSON_PATH.exists()
assert EXCEL_PATH.exists()

print("Synthetic input generation: PASS")


## 3. Load Client XYZ rules

The rules are configuration rather than hard-coded decisions scattered throughout the notebook. This is the beginning of a configurable ingestion workflow.


In [ ]:
rules = load_rules(CONFIG_PATH)
rules


## 4. Inspect the incoming delivery

Always inspect what arrived before editing GIS.


In [ ]:
excel_file = pd.ExcelFile(EXCEL_PATH)
print("Workbook sheets:", excel_file.sheet_names)

incoming = pd.read_excel(
    EXCEL_PATH,
    sheet_name=rules["delivery_sheet"],
)

print("Incoming rows:", len(incoming))
print("Incoming columns:", list(incoming.columns))
incoming.head(10)


### Expected training scenarios

This sheet is our hidden answer key for the synthetic exercise. We use it to verify that the validation engine is classifying records correctly.


In [ ]:
expected_counts = pd.read_excel(
    EXCEL_PATH,
    sheet_name="Expected_Test_Counts",
)
expected_counts


## 5. Convert portable GeoJSON to a local working geodatabase

GeoJSON is the portable source format. ArcGIS editing occurs in a native local feature class.


In [ ]:
arcpy.env.overwriteOutput = True

GDB_PATH = OUTPUT_DIR / "Day01_Working.gdb"
FC_PATH = GDB_PATH / "authoritative_assets"

if not arcpy.Exists(str(GDB_PATH)):
    arcpy.management.CreateFileGDB(str(OUTPUT_DIR), GDB_PATH.name)

if arcpy.Exists(str(FC_PATH)):
    arcpy.management.Delete(str(FC_PATH))

arcpy.conversion.JSONToFeatures(
    str(GEOJSON_PATH),
    str(FC_PATH),
    "POINT",
)

feature_count = int(arcpy.management.GetCount(str(FC_PATH))[0])
print("Working GIS feature count:", feature_count)
assert feature_count == 1000


## 6. Read authoritative GIS attributes

The authoritative GIS dataset represents the organization's current accepted state. Incoming client records are reconciled against it.


In [ ]:
authoritative = read_feature_class(FC_PATH)

print("Authoritative rows:", len(authoritative))
authoritative.head()


## 7. Run the validation and reconciliation engine

This single step applies the configured organizational standards and classifies every incoming record.


In [ ]:
classified, audit, summary = reconcile_delivery(
    incoming=incoming,
    authoritative=authoritative,
    rules=rules,
)

summary


## 8. Review classification results


In [ ]:
status_counts = (
    classified["validation_status"]
    .value_counts()
    .rename_axis("validation_status")
    .reset_index(name="record_count")
)
status_counts


### Verify our engine against the synthetic answer key

The detailed expected scenario labels collapse into four production outcomes:

- `READY_TO_UPDATE`
- `NO_CHANGE`
- `REJECTED`
- `UNMATCHED`


In [ ]:
verification = pd.crosstab(
    classified["scenario_expected"],
    classified["validation_status"],
)
verification


In [ ]:
assert summary["records_received"] == 300
assert summary["ready_to_update"] == 210
assert summary["no_change"] == 20
assert summary["rejected"] == 55
assert summary["unmatched"] == 15

print("Expected classification counts: PASS")


## 9. Investigate rejected records

A useful validation tool does not merely say "bad data." It explains **why** each record was rejected.


In [ ]:
rejection_reasons = (
    classified.loc[
        classified["validation_status"].eq("REJECTED"),
        "validation_reason",
    ]
    .value_counts()
    .rename_axis("reason")
    .reset_index(name="records")
)
rejection_reasons


## 10. Inspect unmatched assets

Unmatched client IDs are preserved for investigation rather than silently dropped.


In [ ]:
classified.loc[
    classified["validation_status"].eq("UNMATCHED"),
    ["client_record_id", "asset_id", "validation_reason", "comments"],
].head(20)


## 11. Inspect the field-level change audit

The audit answers: **what field is proposed to change, from what value, to what value?**


In [ ]:
print("Proposed field-level changes:", len(audit))
audit.head(20)


## 12. Split and export QA/QC reports


In [ ]:
reports = split_reports(classified, audit)
report_paths = write_csv_reports(reports, OUTPUT_DIR)

for name, path in report_paths.items():
    print(f"{name}: {path.name}")


## 13. Controlled GIS update

Only records classified as `READY_TO_UPDATE` are allowed to edit the local working feature class.

This is a **local copy**, not a production database.


In [ ]:
ready_records = reports["Ready_to_Update"]

updated_count = apply_updates_to_feature_class(
    feature_class=FC_PATH,
    ready_records=ready_records,
    rules=rules,
)

summary["gis_features_updated"] = updated_count

print("GIS features updated:", updated_count)
assert updated_count == 210


## 14. Verify edits after the update

Never treat successful code execution as proof that the data is correct.


In [ ]:
after = read_feature_class(FC_PATH)

sample_ids = ["AST-0001", "AST-0003", "AST-0221"]
after[after["asset_id"].isin(sample_ids)].sort_values("asset_id")


## 15. Create the Excel QA/QC dashboard

The dashboard translates technical validation results into something a GIS lead, analyst, or project manager can review quickly.


In [ ]:
DASHBOARD_PATH = OUTPUT_DIR / "XYZ_GIS_Data_Validation_Report.xlsx"

generate_dashboard(
    output_path=DASHBOARD_PATH,
    summary=summary,
    reports=reports,
    classified=classified,
)

print("Excel dashboard created:", DASHBOARD_PATH.name)


## 16. Final Day 01 checks

Version 1 is ready for PR review when all of these are true:

- 1,000 authoritative assets are generated
- 300 incoming Client XYZ records are generated
- schema and IDs are validated
- domains, dates, and numeric ranges are enforced
- whitespace/lowercase IDs normalize correctly
- duplicate and missing IDs are rejected
- unmatched IDs are preserved
- stale inspections are rejected
- blank partial-update fields preserve GIS values
- no-change records are separated
- only ready records update the working feature class
- field-level audit records are generated
- CSV reports are generated
- Excel QA/QC dashboard is generated
- notebook runs top-to-bottom from a clean restart


In [ ]:
assert len(authoritative) == 1000
assert len(classified) == 300
assert summary["ready_to_update"] == 210
assert summary["no_change"] == 20
assert summary["rejected"] == 55
assert summary["unmatched"] == 15
assert summary["gis_features_updated"] == 210
assert DASHBOARD_PATH.exists()

print("DAY 01 TECHNICAL VALIDATION: PASS")


## Developer takeaway

This project is not simply "Excel updates GIS."

It implements a **pre-ingestion GIS data quality gate** between a recurring client field-data delivery and the production GIS team.

A concise interview explanation:

> I designed a reusable GIS data-ingestion workflow for recurring client field deliveries. The solution validates schema and business rules, normalizes and reconciles asset IDs against authoritative GIS, separates rejected and unmatched records, detects stale and no-change records, creates a field-level audit trail, applies only approved updates to a controlled working feature class, and produces an Excel QA/QC dashboard for operational review.

## Git checkpoint

After the clean rerun, inspect generated files locally but do not commit `output/` or generated sample inputs. Commit only source code, configuration, notebook, and documentation.
